In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [39]:
num_tokens = 1000
prefill = (torch.rand(num_tokens, 768, device=device) - 0.5)

w_k = torch.rand(768, 768, device=device) - 0.5
w_v = torch.rand(768, 768, device=device) - 0.5
w_q = torch.rand(768, 768, device=device) - 0.5

k = torch.matmul(prefill, w_k)
v = torch.matmul(prefill, w_v)
q = torch.matmul(prefill, w_q)

# ------------------------------------------------------------------
# 2. Fingerprint Keys & Cluster via KMeans
# ------------------------------------------------------------------
# Select 64 random query fingerprints
q_fingerprint = q[torch.randint(0, num_tokens, (64,), device=device)]
scores = torch.matmul(q_fingerprint, k.T)

# Scale scores and run KMeans clustering
scaler = StandardScaler()
scores_scaled = scaler.fit_transform(scores.cpu().numpy())

kmeans = KMeans(n_clusters=64, random_state=0, init="k-means++").fit(scores_scaled.T)
cluster_labels = torch.tensor(kmeans.labels_, device=device)

# ------------------------------------------------------------------
# 3. Vectorized Cluster Means & Key Counts
# ------------------------------------------------------------------
average_keys = torch.zeros(64, 768, device=device)
cluster_counts = torch.zeros(64, device=device)

# Pre-group key tensors by cluster to eliminate filtering inside loops
cluster_keys_list = []
cluster_values_list = []
for i in range(64):
    c_keys = k[cluster_labels == i]
    v_keys = v[cluster_labels == i]
    cluster_keys_list.append(c_keys)
    cluster_values_list.append(v_keys)
    
    average_keys[i] = torch.mean(c_keys, dim=0)
    cluster_counts[i] = c_keys.shape[0]

print(average_keys.shape)

# ------------------------------------------------------------------
# 4. Fully Vectorized Query-Cluster Sampling & Key Extraction
# ------------------------------------------------------------------
# Compute unnormalized probabilities across ALL queries & clusters simultaneously
# (q @ average_keys.T) shape: (1000, 64)
scale = torch.sqrt(torch.tensor(768.0, device=device))
scores_all = torch.matmul(q, average_keys.T) / scale
z_hat_alpha_all = torch.exp(scores_all) * cluster_counts  # Shape: (1000, 64)

# Compute normalized cluster probabilities per query
z_hat_final = z_hat_alpha_all.sum(dim=-1, keepdim=True)
probabilities = z_hat_alpha_all / z_hat_final  # Shape: (1000, 64)

# Sample 64 cluster indices for each of the 1000 queries simultaneously
# Shape: (1000, 64)
sampled_clusters = torch.multinomial(probabilities, num_samples=64, replacement=True)

# retrieve the original keys for the sampled clusters and compute the actual attention with correction factor that is z_hat_alpha / z_hat_final * 1 / cluster_counts
attention_output = torch.zeros(num_tokens, 768, device=device)
for q_idx in range(num_tokens):
    query_sampled_clusters = sampled_clusters[q_idx]
    z_partition = 0
    
    for cluster_id in query_sampled_clusters:
        c_id = cluster_id.item()
        c_keys = cluster_keys_list[c_id]
        
        # Uniformly sample one key from the cluster
        rand_idx = torch.randint(0, c_keys.shape[0], (1,), device=device)
        sampled_key = c_keys[rand_idx]

        v_keys = cluster_values_list[c_id]
        sampled_value = v_keys[rand_idx]
        # Compute the correction factor
        correction_factor = (z_hat_alpha_all[q_idx, c_id] / z_hat_final[q_idx]) * (1.0 / cluster_counts[c_id])

        score = torch.matmul(q[q_idx], sampled_key.T)

        z_partition += score / correction_factor
        
        # Compute attention output for this query and sampled key-value pair
        attention_output[q_idx, :] += torch.squeeze(score * sampled_value / correction_factor)

    attention_output[q_idx, :] /= z_partition

print(attention_output.shape)

c:\CondaEnvs\opus-ai\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


torch.Size([64, 768])
torch.Size([1000, 768])
